# CHIT No. 26 — Scrape a Permitted Website, Clean, Save and Plot

Uses the practice site books.toscrape.com, designed for web-scraping practice. Checks its robots.txt before requesting the catalogue page, extracts a table-like list of books, cleans the price, saves CSV and plots prices. If robots.txt cannot be retrieved or disallows the catalogue, stop rather than scrape.

**How to use:** Run each cell from top to bottom. Read the short explanation before every code cell.

### What this cell does
Import the libraries needed for web requests, HTML parsing, dataframes and charts.

In [ ]:
import requests
import pandas as pd
import matplotlib.pyplot as plt
from bs4 import BeautifulSoup
from urllib.parse import urljoin

### What this cell does
Check the site's robots.txt. This notebook only proceeds if the robots file is accessible and does not disallow the catalogue path for the default user-agent.

In [ ]:
robots_url = "https://books.toscrape.com/robots.txt"
robots_response = requests.get(robots_url, timeout=20)

print("robots.txt status:", robots_response.status_code)
print(robots_response.text[:1500])

if robots_response.status_code != 200:
    raise RuntimeError("Could not verify robots.txt. Stop and check the site's scraping rules before continuing.")

robots_text = robots_response.text.lower()
if "disallow: /" in robots_text or "disallow: /catalogue/" in robots_text:
    raise RuntimeError("robots.txt appears to disallow the catalogue. Do not continue scraping.")

### What this cell does
Request the first catalogue page and parse the book listing.

In [ ]:
url = "https://books.toscrape.com/"
response = requests.get(url, timeout=20)
response.raise_for_status()
soup = BeautifulSoup(response.text, "html.parser")
books = soup.select("article.product_pod")
print("Books found:", len(books))

### What this cell does
Extract each book's title, price and rating into a list, then convert it to a dataframe.

In [ ]:
rating_map = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}
rows = []

for book in books:
    rating_word = book.select_one("p.star-rating")["class"][1]
    rows.append({
        "Title": book.h3.a["title"],
        "Price": book.select_one(".price_color").get_text(strip=True),
        "Rating": rating_map.get(rating_word)
    })

df = pd.DataFrame(rows)
df.head()

### What this cell does
Remove the pound sign from Price and convert the column to a number.

In [ ]:
df["Price"] = df["Price"].str.replace("£", "", regex=False).str.replace("Â", "", regex=False).astype(float)
print(df.dtypes)
display(df.head())

### What this cell does
Save the cleaned data to CSV and print the file name.

In [ ]:
output_file = "scraped_books_clean.csv"
df.to_csv(output_file, index=False)
print("Saved:", output_file)
print("Rows saved:", len(df))

### What this cell does
Plot a bar chart showing the prices of the first ten books.

In [ ]:
plot_data = df.head(10)

plt.figure(figsize=(11, 5))
plt.bar(plot_data["Title"], plot_data["Price"])
plt.title("Prices of the First 10 Books")
plt.xlabel("Book title")
plt.ylabel("Price (£)")
plt.xticks(rotation=75, ha="right")
plt.tight_layout()
plt.show()